# 03 — Gold: bid performance

Business-facing aggregates. Two rules shape this layer:

**Every rate is reported twice** — including and excluding bulk-loaded
records. A single number here would be misleading, and which one is "correct"
depends on the question being asked.

**Win rate is reported by count and by value.** The two differ by roughly five
points, and only the value-weighted figure reflects commercial reality.

Sales-cycle length is deliberately absent. More than half the closure
timestamps were written in bulk-closing sessions, seconds apart, months after
the fact — any duration derived from them would be fiction.

In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS gold.bid")

bids = spark.table("silver.bid.bids_clean")
clients = spark.table("silver.bid.clients_clean")

fact = (
    bids.join(F.broadcast(clients), "client_id", "left")
        .filter(F.col("outcome").isNotNull())      # closed bids only
)

## Core metrics

`win_rate_by_count` treats a R$20k bid and a R$2M bid identically.
`win_rate_by_value` does not. Publishing only the first overstates
performance.

In [0]:
def performance(df, *dims):
    return (
        df.groupBy(*dims)
          .agg(
              F.count("*").alias("bids_closed"),
              F.sum("outcome").alias("bids_won"),
              F.round(F.avg("outcome") * 100, 1).alias("win_rate_by_count"),
              F.round(
                  F.sum(F.when(F.col("outcome") == 1, F.col("contract_value_brl")).otherwise(0))
                  / F.sum("contract_value_brl") * 100, 1
              ).alias("win_rate_by_value"),
              F.round(F.sum("contract_value_brl"), 2).alias("value_bid_brl"),
          )
    )


overall = performance(fact, F.lit(True).alias("_all")).drop("_all")
by_channel = performance(fact, "is_bulk_load")

overall.write.format("delta").mode("overwrite").saveAsTable("gold.bid.performance_overall")
by_channel.write.format("delta").mode("overwrite").saveAsTable("gold.bid.performance_by_channel")

display(by_channel)

## Segmentation, with the confound isolated

Each dimension is reported on all closed bids and again on organic bids only.
The gap between the two columns is the size of the migration artefact — and
for at least one executive it is the difference between "underperforming" and
"above average".

In [0]:
organic = fact.filter(~F.col("is_bulk_load"))

for dim in ["segment", "state", "account_executive"]:
    combined = (
        performance(fact, dim)
        .select(dim, "bids_closed", F.col("win_rate_by_count").alias("wr_all"))
        .join(
            performance(organic, dim)
              .select(dim,
                      F.col("bids_closed").alias("bids_organic"),
                      F.col("win_rate_by_count").alias("wr_organic")),
            dim, "left",
        )
        .withColumn("artefact_gap", F.round(F.col("wr_organic") - F.col("wr_all"), 1))
        .orderBy(F.col("bids_closed").desc())
    )
    combined.write.format("delta").mode("overwrite").saveAsTable(f"gold.bid.performance_by_{dim}")
    display(combined)

## Contract value effect

Quartiles are computed across closed bids. If the win rate falls as value
rises, count-based reporting is systematically flattering.

Quartile boundaries come from `approxQuantile`, not `ntile()` over an
unpartitioned window. `ntile` needs every row sorted on one node to assign
exact ranks — harmless at 1,600 rows, but it's the pattern behind the
`WindowExpression: No Partition Defined` warning Spark throws, and it stops
scaling long before the rest of this pipeline would. `approxQuantile` finds
the three cut points with a distributed, sketch-based algorithm and then
buckets every row with a plain `when/otherwise` — no shuffle-to-one-node
step at all.

In [0]:
# Approximate quartile boundaries (relativeError=0.01 — plenty tight for
# reporting bands, and cheap regardless of table size).
q1, q2, q3 = fact.approxQuantile("contract_value_brl", [0.25, 0.5, 0.75], 0.01)

value_bands = fact.withColumn(
    "value_quartile",
    F.when(F.col("contract_value_brl") <= q1, 1)
     .when(F.col("contract_value_brl") <= q2, 2)
     .when(F.col("contract_value_brl") <= q3, 3)
     .otherwise(4),
)

by_value = (
    performance(value_bands, "value_quartile")
    .orderBy("value_quartile")
)

by_value.write.format("delta").mode("overwrite").saveAsTable("gold.bid.performance_by_value_band")
display(by_value)

## Loss reasons, and how little of the picture they cover

The coverage figure is the point of this table. A reason distribution built
on a single-digit share of losses is a signal, not a population estimate, and
the two must be published together.

`share_pct` used to divide by a sum computed with `Window.partitionBy()` —
an empty partition spec, same problem as the quartiles above: it forces the
whole DataFrame onto one node just to get a single total. With only a
handful of distinct reasons, the total is one number — computed once with
`.collect()` and reused as a plain scalar.

In [0]:
losses = bids.filter(F.col("outcome") == 0)

coverage = losses.agg(
    F.count("*").alias("losses_total"),
    F.sum(F.col("has_loss_reason").cast("int")).alias("losses_with_reason"),
    F.round(F.avg(F.col("has_loss_reason").cast("int")) * 100, 1).alias("coverage_pct"),
    F.sum(F.col("competitor_is_placeholder").cast("int")).alias("placeholder_attributed"),
)

reasons_raw = (
    losses.filter(F.col("has_loss_reason"))
          .groupBy("loss_reason")
          .agg(F.count("*").alias("losses"))
)

# One number, computed once — cheaper and clearer than a windowed sum.
total_with_reason = reasons_raw.agg(F.sum("losses")).collect()[0][0]

reasons = (
    reasons_raw
    .withColumn("share_pct", F.round(F.col("losses") / F.lit(total_with_reason) * 100, 1))
    .orderBy(F.col("losses").desc())
)

coverage.write.format("delta").mode("overwrite").saveAsTable("gold.bid.loss_reason_coverage")
reasons.write.format("delta").mode("overwrite").saveAsTable("gold.bid.loss_reasons")

display(coverage)
display(reasons)

## Open pipeline

Bids with a NULL outcome. Reported separately so they never silently join the
loss column, which would understate the win rate by roughly a third.

In [0]:
pipeline = (
    bids.filter(F.col("outcome").isNull())
        .join(F.broadcast(clients), "client_id", "left")
        .groupBy("segment")
        .agg(
            F.count("*").alias("bids_open"),
            F.round(F.sum("contract_value_brl"), 2).alias("value_open_brl"),
        )
        .orderBy(F.col("value_open_brl").desc())
)

pipeline.write.format("delta").mode("overwrite").saveAsTable("gold.bid.open_pipeline")
display(pipeline)